# 02 — Chunking Demo

Full pipeline: **PDF → extract blocks → classify → chunk**

This notebook demonstrates Layers 1 → 2 → 4 of the HSC-Edu pipeline,
ending with semantic chunks ready for embedding.

In [1]:
from pathlib import Path

from hsc_edu.core.extraction import extract_document
from hsc_edu.core.classification import classify_blocks
from hsc_edu.core.chunking import chunk_blocks, count_tokens
from hsc_edu.core.models import BlockType

DATA_DIR = Path("../data")
SAMPLE_PDF = DATA_DIR / "Java.pdf"

print(f"PDF: {SAMPLE_PDF.name}  ({SAMPLE_PDF.stat().st_size / 1024 / 1024:.1f} MB)")

PDF: Java.pdf  (6.3 MB)


## 1. Extract blocks (Layer 1)

In [2]:
blocks = extract_document(SAMPLE_PDF, doc_id="java-demo")
print(f"Extracted {len(blocks)} blocks from '{SAMPLE_PDF.name}'")

Extracted 2121 blocks from 'Java.pdf'


## 2. Classify blocks (Layer 2)

In [3]:
classified = classify_blocks(blocks)

from collections import Counter

type_counts = Counter(b.block_type for b in classified)
print(f"Classified {len(classified)} blocks:")
for bt, cnt in type_counts.most_common():
    print(f"  {bt:<20s} {cnt:>5d}")

Classified 2121 blocks:
  paragraph             1782
  heading                325
  exercise                13
  note                     1


In [4]:
headings = [b for b in classified if b.block_type == BlockType.HEADING]
print(f"Headings found: {len(headings)}\n")

for h in headings[:15]:
    indent = "  " * ((h.heading_level or 1) - 1)
    text = h.raw_text.split("\n")[0][:80]
    print(f"  p.{h.page:>3d}  lv{h.heading_level}  {indent}{text}")

Headings found: 325

  p.  0  lv1  Chương 1. MỞ ĐẦU ............................................................7
  p.  0  lv2    1.1. KHÁI NIỆM CƠ BẢN ................................................ 12
  p.  0  lv2    1.2. ĐỐI TƯỢNG VÀ LỚP................................................ 13
  p.  0  lv2    1.3. CÁC NGUYÊN TẮC TRỤ CỘT ................................ 15
  p.  0  lv1  Chương 2. NGÔN NGỮ LẬP TRÌNH JAVA ................... 20
  p.  0  lv2    2.1. ĐẶC TÍNH CỦA JAVA .............................................. 20
  p.  0  lv3      2.1.1. Máy ảo Java – Java Virtual Machine ............... 21
  p.  0  lv3      2.1.2. Các nền tảng Java ............................................. 23
  p.  0  lv3      2.1.3. Môi trường lập trình Java ................................ 23
  p.  0  lv3      2.1.4. Cấu trúc mã nguồn Java .................................. 24
  p.  0  lv3      2.1.5. Chương trình Java đầu tiên ............................. 25
  p.  0  lv2    2.2. BIẾN ............

## 3. Chunk (Layer 4)

In [5]:
chunks = chunk_blocks(classified)

print(f"Total chunks: {len(chunks)}")
print(f"Token range : {min(c.token_count for c in chunks)} – {max(c.token_count for c in chunks)}")
print(f"Avg tokens  : {sum(c.token_count for c in chunks) / len(chunks):.0f}")

Total chunks: 434
Token range : 18 – 1064
Avg tokens  : 427


### 3.1 Token distribution

In [6]:
buckets = {"0-64": 0, "64-256": 0, "256-512": 0, "512-1024": 0, ">1024": 0}
for c in chunks:
    t = c.token_count
    if t <= 64:
        buckets["0-64"] += 1
    elif t <= 256:
        buckets["64-256"] += 1
    elif t <= 512:
        buckets["256-512"] += 1
    elif t <= 1024:
        buckets["512-1024"] += 1
    else:
        buckets[">1024"] += 1

print("Token distribution:")
for label, cnt in buckets.items():
    bar = "█" * cnt
    print(f"  {label:>10s}: {cnt:>4d}  {bar}")

Token distribution:
        0-64:  131  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
      64-256:   89  █████████████████████████████████████████████████████████████████████████████████████████
     256-512:   35  ███████████████████████████████████
    512-1024:  171  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
       >1024:    8  ████████


## 4. First 10 chunks (with section path)

Each chunk shows its **heading hierarchy** (`section_path`) so you can see the table-of-contents context.

In [7]:
SHOW_N = 10

for i, ch in enumerate(chunks[:SHOW_N]):
    path_str = " > ".join(ch.section_path) if ch.section_path else "(no heading)"
    text_preview = ch.text[:200].replace("\n", " ↵ ")
    if len(ch.text) > 200:
        text_preview += " …"

    print(f"{'═' * 72}")
    print(f"Chunk {i}  │  {ch.token_count} tokens  │  pages {ch.page_start}–{ch.page_end}  │  type: {ch.block_type}")
    print(f"Section: {path_str}")
    print(f"Text   : {text_preview}")
    print()

════════════════════════════════════════════════════════════════════════
Chunk 0  │  18 tokens  │  pages 0–0  │  type: paragraph
Section: (no heading)
Text   : Mục lục ↵  ↵ GIỚI THIỆU .............................................................................5

════════════════════════════════════════════════════════════════════════
Chunk 1  │  33 tokens  │  pages 0–0  │  type: paragraph
Section: Chương 1. MỞ ĐẦU ............................................................7
Text   : Chương 1. MỞ ĐẦU ............................................................7 ↵  ↵ Chương 1. MỞ ĐẦU ............................................................7

════════════════════════════════════════════════════════════════════════
Chunk 2  │  45 tokens  │  pages 0–0  │  type: paragraph
Section: Chương 1. MỞ ĐẦU ............................................................7 > 1.1. KHÁI NIỆM CƠ BẢN ................................................ 12
Text   : 1.1. KHÁI NIỆM CƠ BẢN ......................

## 5. Inspect a specific chunk

Change `CHUNK_IDX` to drill into any chunk.

In [15]:
CHUNK_IDX = 150  # ← change this

ch = chunks[CHUNK_IDX]

print(f"chunk_id     : {ch.chunk_id}")
print(f"doc_id       : {ch.doc_id}")
print(f"block_type   : {ch.block_type}")
print(f"token_count  : {ch.token_count}")
print(f"pages        : {ch.page_start} – {ch.page_end}")
print(f"chapter      : {ch.chapter}")
print(f"section      : {ch.section}")
print(f"section_path : {ch.section_path}")
print(f"block_ids    : {ch.block_ids}")
print(f"\n{'─' * 72}")
print(ch.text)

chunk_id     : ea67e2437174
doc_id       : java-demo
block_type   : paragraph
token_count  : 737
pages        : 28 – 29
chapter      : Chương 13. LẬP TRÌNH TỔNG QUÁT VÀ CÁC LỚP COLLECTION
section      : 2.3.3. Các phép toán khác
section_path : ['Chương 13. LẬP TRÌNH TỔNG QUÁT VÀ CÁC LỚP COLLECTION', '2.3. CÁC PHÉP TOÁN CƠ BẢN', '2.3.3. Các phép toán khác']
block_ids    : ['6020682bd07b', 'b3b89b1be760', 'ec3b1fd284ef', '7b7e8990ca2a', '64279befc4b5', '128659dfdff4', 'd5f028744998', '5211a8db762e', '6471aa249593', '22efda8901b0', '14ae69f0d5bc', '5c5591ba96a5', '2b324ae6a42b', 'f38cc4a1a0c3', '074970c782f8', '5ed8b91d0958', '22f8c0f94f62', '54ae4ff1e434', '33c49895bb3a', 'abd3179460f6', '6cdeba951b18', 'e5f47c1f3fa5']

────────────────────────────────────────────────────────────────────────
2.3.3. Các phép toán khác

2.3.3. Các phép toán khác

Các phép toán so sánh được sử dụng để so sánh giá trị hai biểu thức. Các phép 
toán này cho kết quả kiểu boolean bằng true nếu đúng và false nếu 